# 🏥 Healthcare QA Chatbot - Build Knowledge Base

This notebook builds the medical knowledge base using Google Colab's free GPU.

**Steps:**
1. Install dependencies
2. Download medical datasets from HuggingFace
3. Build vector embeddings
4. Download the knowledge base to your local machine

## 1️⃣ Install Dependencies

In [ ]:
!pip install -q chromadb==0.4.24 sentence-transformers datasets pandas pyarrow tqdm

## 2️⃣ Download Medical Datasets

In [ ]:
import os
from pathlib import Path
from datasets import load_dataset
import pandas as pd

# Create directories
DATA_DIR = Path("/content/data/raw")
KB_DIR = Path("/content/data/knowledge_base")
DATA_DIR.mkdir(parents=True, exist_ok=True)
KB_DIR.mkdir(parents=True, exist_ok=True)

print("Downloading datasets...")

# MedQuAD
try:
    ds = load_dataset("keivalya/MedQuad-MedicalQnADataset", split="train")
    df = ds.to_pandas()
    (DATA_DIR / "mediqa").mkdir(exist_ok=True)
    df.to_parquet(DATA_DIR / "mediqa" / "medquad.parquet")
    print(f"✅ MedQuAD: {len(df):,} samples")
except Exception as e:
    print(f"❌ MedQuAD: {e}")

# PubMedQA
try:
    ds = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")
    df = ds.to_pandas()
    (DATA_DIR / "pubmed").mkdir(exist_ok=True)
    df.to_parquet(DATA_DIR / "pubmed" / "pubmedqa_labeled.parquet")
    print(f"✅ PubMedQA: {len(df):,} samples")
except Exception as e:
    print(f"❌ PubMedQA: {e}")

# MedMCQA (limit to 50k for speed)
try:
    ds = load_dataset("openlifescienceai/medmcqa", split="train[:50000]")
    df = ds.to_pandas()
    df.to_parquet(DATA_DIR / "mediqa" / "medmcqa_train.parquet")
    print(f"✅ MedMCQA: {len(df):,} samples")
except Exception as e:
    print(f"❌ MedMCQA: {e}")

# HealthCareMagic (limit to 30k for speed)
try:
    ds = load_dataset("truehealth/healthcaremagic", split="train[:30000]")
    df = ds.to_pandas()
    df.to_parquet(DATA_DIR / "mediqa" / "healthcare_magic.parquet")
    print(f"✅ HealthCareMagic: {len(df):,} samples")
except Exception as e:
    print(f"❌ HealthCareMagic: {e}")

print("\n✅ Dataset download complete!")

## 3️⃣ Build Knowledge Base

In [ ]:
import gc
import numpy as np
from typing import List, Dict, Generator
from tqdm.notebook import tqdm
from sentence_transformers import SentenceTransformer
import chromadb

# Initialize embedding model (uses GPU automatically in Colab)
print("Loading embedding model...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
print(f"✅ Model loaded. Dimension: {embedder.get_sentence_embedding_dimension()}")

# Initialize vector store
print("Initializing vector store...")
client = chromadb.PersistentClient(path=str(KB_DIR))
collection = client.get_or_create_collection(
    name="medical_knowledge",
    metadata={"hnsw:space": "cosine"}
)
print(f"✅ Vector store ready")

In [ ]:
def chunk_text(text: str, chunk_size: int = 512, overlap: int = 50) -> List[str]:
    """Split text into chunks."""
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        if chunk.strip():
            chunks.append(chunk)
        start = end - overlap
        if end >= len(words):
            break
    return chunks if chunks else [text]

def load_qa_pairs() -> Generator[Dict, None, None]:
    """Load QA pairs from parquet files."""
    # MedQuAD
    path = DATA_DIR / "mediqa" / "medquad.parquet"
    if path.exists():
        df = pd.read_parquet(path)
        for _, row in df.iterrows():
            yield {
                "question": row.get("Question", row.get("question", "")),
                "answer": row.get("Answer", row.get("answer", "")),
                "source": "MedQuAD"
            }
        print(f"  Loaded MedQuAD: {len(df):,}")

    # PubMedQA
    path = DATA_DIR / "pubmed" / "pubmedqa_labeled.parquet"
    if path.exists():
        df = pd.read_parquet(path)
        for _, row in df.iterrows():
            yield {
                "question": row.get("question", ""),
                "answer": row.get("long_answer", ""),
                "source": "PubMedQA"
            }
        print(f"  Loaded PubMedQA: {len(df):,}")

    # MedMCQA
    path = DATA_DIR / "mediqa" / "medmcqa_train.parquet"
    if path.exists():
        df = pd.read_parquet(path)
        count = 0
        for _, row in df.iterrows():
            answer = row.get("exp")
            if answer and not pd.isna(answer):
                yield {
                    "question": row.get("question", ""),
                    "answer": str(answer),
                    "source": "MedMCQA"
                }
                count += 1
        print(f"  Loaded MedMCQA: {count:,}")

    # HealthCareMagic
    path = DATA_DIR / "mediqa" / "healthcare_magic.parquet"
    if path.exists():
        df = pd.read_parquet(path)
        for _, row in df.iterrows():
            question = row.get("input", row.get("instruction", ""))
            yield {
                "question": question,
                "answer": row.get("output", ""),
                "source": "HealthCareMagic"
            }
        print(f"  Loaded HealthCareMagic: {len(df):,}")

# Process all documents
print("\n📚 Processing documents...")
all_chunks = []
doc_count = 0

for qa in tqdm(load_qa_pairs(), desc="Loading QA pairs"):
    content = f"Question: {qa['question']}\n\nAnswer: {qa['answer']}"
    if len(content.strip()) < 50:
        continue
    
    chunks = chunk_text(content)
    for i, chunk in enumerate(chunks):
        all_chunks.append({
            "content": chunk,
            "source": qa["source"],
            "chunk_id": i + 1,
            "total_chunks": len(chunks)
        })
    doc_count += 1
    
    if doc_count % 20000 == 0:
        gc.collect()
        print(f"  Processed {doc_count:,} documents, {len(all_chunks):,} chunks")

print(f"\n✅ Total documents: {doc_count:,}")
print(f"✅ Total chunks: {len(all_chunks):,}")

In [ ]:
# Generate embeddings and index
print("\n🔢 Generating embeddings and indexing...")
batch_size = 500
total = len(all_chunks)

for i in tqdm(range(0, total, batch_size), desc="Indexing"):
    batch = all_chunks[i:i + batch_size]
    texts = [c["content"] for c in batch]
    
    try:
        # Generate embeddings
        embeddings = embedder.encode(texts, batch_size=64, show_progress_bar=False)
        
        # Prepare metadata
        metadatas = [
            {
                "source": c["source"],
                "chunk_id": c["chunk_id"],
                "total_chunks": c["total_chunks"]
            }
            for c in batch
        ]
        
        ids = [f"chunk_{i + j}" for j in range(len(batch))]
        
        # Add to collection
        collection.add(
            ids=ids,
            embeddings=embeddings.tolist(),
            documents=texts,
            metadatas=metadatas
        )
    except Exception as e:
        print(f"Error at batch {i}: {e}")
        continue
    
    if (i // batch_size) % 50 == 0:
        gc.collect()

print(f"\n✅ Indexed {collection.count():,} chunks!")

## 4️⃣ Download Knowledge Base

Run the cell below to download the knowledge base as a zip file.

In [ ]:
import shutil
from google.colab import files

print("📦 Creating zip archive...")
shutil.make_archive("/content/knowledge_base", 'zip', KB_DIR)

print("📥 Downloading...")
files.download("/content/knowledge_base.zip")

print("\n✅ Download complete!")
print("\n📋 Next steps:")
print("1. Extract knowledge_base.zip")
print("2. Replace your local data/knowledge_base/ folder with the extracted contents")
print("3. Restart your API server")

## 🧪 Test the Knowledge Base (Optional)

In [ ]:
# Test query
test_query = "What are the symptoms of diabetes?"
query_embedding = embedder.encode([test_query])[0].tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

print(f"Query: {test_query}\n")
print("Top 3 results:")
for i, (doc, meta) in enumerate(zip(results['documents'][0], results['metadatas'][0])):
    print(f"\n--- Result {i+1} (Source: {meta['source']}) ---")
    print(doc[:500] + "..." if len(doc) > 500 else doc)